# Лабораторная работа №2. Логистическая и линейная регрессия

Чибугаев Илья Алексеевич, вариант 22.

## Данные и метрики

Для классификации взят [Ionosphere](https://archive.ics.uci.edu/dataset/52/ionosphere): 351 радиосигнал и 34 признака. Класс `bad=1` означает сигнал без структуры в ионосфере. Это задача автоматической проверки радиосигналов.

Для регрессии взят [Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength): 1030 составов бетона и 8 признаков. По составу и возрасту нужно оценить прочность в МПа. Такой прогноз полезен перед лабораторными испытаниями смеси.

Логистическая регрессия возвращает вероятность класса через сигмоиду. Линейная регрессия минимизирует квадраты ошибок, а полиномиальные признаки позволяют учесть нелинейную связь состава бетона и прочности.

Train и test соотносятся как 80/20, `random_state=22`. Классы при разбиении сохраняются в одинаковой пропорции. Подбор параметров идет только на train.

В классификации основной показатель: ROC-AUC. Он не зависит от выбранного порога. F1 показывает качество для `bad`, accuracy дает общую долю верных ответов. В регрессии основной показатель: RMSE в МПа. MAE проще интерпретировать как типичную ошибку, а R² показывает долю объясненной дисперсии.


## Подготовка

Общий код вынесен в `src`, чтобы не повторять его в каждой работе.

In [1]:
# Подключаю общие функции и настраиваю вывод таблиц.
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.experiments import TaskExperiment

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Корень проекта: {ROOT}")

Корень проекта: /Users/finleyl/Documents/kimi/tasks/2026-11-10/20-02-04-6933ea61/ml-labs-1-5


## Проверка данных

Смотрю размеры выборок, пропуски и целевые переменные. Эти разбиения используются во всех работах.

In [2]:
# Загружаю данные и проверяю размеры выборок.
experiments = {task: TaskExperiment(2, task) for task in ("classification", "regression")}
for task, experiment in experiments.items():
    data = experiment.data
    print(f"{task}: {data.name}")
    print(data.practical_task)
    print(f"train={data.X_train.shape}, test={data.X_test.shape}, пропуски={int(data.X.isna().sum().sum())}")
    if task == "classification":
        print(f"Доля bad: train={data.y_train.mean():.3f}, test={data.y_test.mean():.3f}")
    else:
        print(data.y.describe().to_string())
    print()

classification: Ionosphere (UCI)
По сигналам фазированной антенной решетки отличать радиосигналы со структурой в ионосфере от проходящих сигналов.
train=(280, 34), test=(71, 34), пропуски=0
Доля bad: train=0.361, test=0.352

regression: Concrete Compressive Strength (UCI)
По составу смеси и возрасту бетона прогнозировать прочность на сжатие в МПа для предварительной оценки рецептуры.
train=(824, 8), test=(206, 8), пропуски=0
count    1030.0000
mean       35.8178
std        16.7057
min         2.3318
25%        23.7071
50%        34.4428
75%        46.1363
max        82.5992



## Бейзлайн sklearn

Беру модель с параметрами по умолчанию. Медианный imputer оставлен как страховка, хотя пропусков в данных нет.

In [3]:
# Обучаю базовые модели sklearn.
for experiment in experiments.values():
    experiment.fit_sklearn_baseline()
baseline_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(baseline_results.drop(columns=["params"]))

classification | sklearn | baseline: accuracy=0.7746, f1=0.5294, roc_auc=0.8313; 0.01 s
regression | sklearn | baseline: rmse=10.1625, mae=8.1968, r2=0.6542; 0.00 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,2,Логистическая / линейная регрессия,classification,sklearn,baseline,0.7746,0.5294,0.8313,NaN,NaN,NaN,0.0052
1,2,Логистическая / линейная регрессия,regression,sklearn,baseline,NaN,NaN,NaN,10.1625,8.1968,0.6542,0.0019


## Подбор параметров

Для классификации проверяю масштабирование, L2-регуляризацию и балансировку классов. Для бетона сравниваю исходные признаки с полиномом второй степени.

Использую пятикратную CV. Для классификации выбираю по ROC-AUC, для регрессии по RMSE.

In [4]:
# Подбираю параметры только на обучающей выборке.
for experiment in experiments.values():
    experiment.fit_sklearn_improved()
    print(f"{{experiment.task}}: {{experiment.search.best_params_}}")
    display(experiment.cv_table().head())

classification | sklearn | improved: accuracy=0.8028, f1=0.6316, roc_auc=0.8696; 1.83 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_roc_auc,std_roc_auc,mean_fit_time
0,"{'model__C': 0.2, 'model__class_weight': 'bala...",0.9375,0.0061,0.0070
1,"{'model__C': 0.2, 'model__class_weight': None}",0.9353,0.0077,0.0162
2,"{'model__C': 0.05, 'model__class_weight': 'bal...",0.9320,0.0145,0.0236
3,"{'model__C': 0.05, 'model__class_weight': None}",0.9312,0.0162,0.0263
4,"{'model__C': 1.0, 'model__class_weight': None}",0.9208,0.0279,0.0152


regression | sklearn | improved: rmse=7.5310, mae=5.8169, r2=0.8101; 0.03 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_rmse,std_rmse,mean_fit_time
0,{'poly__degree': 2},7.7647,0.4435,0.0030
1,{'poly__degree': 1},10.5173,0.4121,0.0026


## Что дала настройка

Сравниваю метрики до и после. Для ROC-AUC, F1, accuracy и R² рост означает улучшение, для RMSE и MAE наоборот.

In [5]:
# Считаю разницу метрик между базовой и настроенной моделями.
metric_names = {"classification": ["roc_auc", "f1", "accuracy"], "regression": ["rmse", "mae", "r2"]}
for task, experiment in experiments.items():
    rows = experiment.results.set_index("stage")
    metrics = metric_names[task]
    comparison = rows.loc[["baseline", "improved"], metrics].T
    comparison["change"] = comparison["improved"] - comparison["baseline"]
    print(task)
    display(comparison)

classification


stage,baseline,improved,change
roc_auc,0.8313,0.8696,0.0383
f1,0.5294,0.6316,0.1022
accuracy,0.7746,0.8028,0.0282


regression


stage,baseline,improved,change
rmse,10.1625,7.5310,-2.6315
mae,8.1968,5.8169,-2.3800
r2,0.6542,0.8101,0.1559


## Своя реализация

Логистическая регрессия обучается градиентным спуском. Для линейной используется `np.linalg.lstsq`: так устойчивее, чем явное обращение матрицы.

Код лежит в `src/models/`. Сначала обучаю базовую версию на тех же данных, что и sklearn.

In [6]:
# Обучаю базовые версии собственных моделей.
for experiment in experiments.values():
    experiment.fit_custom_baseline()
custom_baseline_results = pd.concat([
    experiment.results[experiment.results["implementation"] == "from_scratch"]
    for experiment in experiments.values()
], ignore_index=True)
display(custom_baseline_results.drop(columns=["params"]))

classification | from_scratch | baseline: accuracy=0.7606, f1=0.4848, roc_auc=0.6730; 0.07 s
regression | from_scratch | baseline: rmse=10.1625, mae=8.1968, r2=0.6542; 0.00 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,2,Логистическая / линейная регрессия,classification,from_scratch,baseline,0.7606,0.4848,0.673,NaN,NaN,NaN,0.0663
1,2,Логистическая / линейная регрессия,regression,from_scratch,baseline,NaN,NaN,NaN,10.1625,8.1968,0.6542,0.0002


## Своя модель после настройки

Переношу препроцессинг и параметры, выбранные на CV. Ограничение по числу деревьев для своего леса указано выше.

In [7]:
# Применяю найденные параметры к своим моделям.
for experiment in experiments.values():
    experiment.fit_custom_improved()
all_lab_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(all_lab_results.drop(columns=["params"]))

classification | from_scratch | improved: accuracy=0.8028, f1=0.6316, roc_auc=0.8704; 0.03 s
regression | from_scratch | improved: rmse=7.5310, mae=5.8169, r2=0.8101; 0.00 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,2,Логистическая / линейная регрессия,classification,sklearn,baseline,0.7746,0.5294,0.8313,NaN,NaN,NaN,0.0052
1,2,Логистическая / линейная регрессия,classification,sklearn,improved,0.8028,0.6316,0.8696,NaN,NaN,NaN,1.8341
2,2,Логистическая / линейная регрессия,classification,from_scratch,baseline,0.7606,0.4848,0.6730,NaN,NaN,NaN,0.0663
3,2,Логистическая / линейная регрессия,classification,from_scratch,improved,0.8028,0.6316,0.8704,NaN,NaN,NaN,0.0306
4,2,Логистическая / линейная регрессия,regression,sklearn,baseline,NaN,NaN,NaN,10.1625,8.1968,0.6542,0.0019
5,2,Логистическая / линейная регрессия,regression,sklearn,improved,NaN,NaN,NaN,7.5310,5.8169,0.8101,0.0296
6,2,Логистическая / линейная регрессия,regression,from_scratch,baseline,NaN,NaN,NaN,10.1625,8.1968,0.6542,0.0002
7,2,Логистическая / линейная регрессия,regression,from_scratch,improved,NaN,NaN,NaN,7.5310,5.8169,0.8101,0.0004


## Результаты

Сохраняю все четыре варианта для каждой задачи. Вывод зависит от полученных чисел: если настройка не помогла, это будет видно.

In [8]:
# Сохраняю таблицу и вывожу изменение основной метрики.
output_path = ROOT / "results" / "lab02_results.csv"
output_path.parent.mkdir(exist_ok=True)
all_lab_results.to_csv(output_path, index=False)

def verdict(change, higher_is_better):
    if np.isclose(change, 0):
        return "не изменился"
    improved = change > 0 if higher_is_better else change < 0
    return "улучшился" if improved else "ухудшился"

for task, experiment in experiments.items():
    frame = experiment.results.set_index(["implementation", "stage"])
    metrics = metric_names[task]
    primary = "roc_auc" if task == "classification" else "rmse"
    higher_is_better = primary != "rmse"
    sklearn_base = frame.loc[("sklearn", "baseline"), primary]
    sklearn_best = frame.loc[("sklearn", "improved"), primary]
    custom_base = frame.loc[("from_scratch", "baseline"), primary]
    custom_best = frame.loc[("from_scratch", "improved"), primary]
    print(f"{task}, sklearn: {primary} {sklearn_base:.4f} → {sklearn_best:.4f}, "
          f"{verdict(sklearn_best - sklearn_base, higher_is_better)}.")
    print(f"{task}, своя версия: {primary} {custom_base:.4f} → {custom_best:.4f}, "
          f"{verdict(custom_best - custom_base, higher_is_better)}.")
    print(f"Параметры sklearn: {experiment.search.best_params_}")
print(f"Файл: {output_path.relative_to(ROOT)}")

classification, sklearn: roc_auc 0.8313 → 0.8696, улучшился.
classification, своя версия: roc_auc 0.6730 → 0.8704, улучшился.
Параметры sklearn: {'model__C': 0.2, 'model__class_weight': 'balanced'}
regression, sklearn: rmse 10.1625 → 7.5310, улучшился.
regression, своя версия: rmse 10.1625 → 7.5310, улучшился.
Параметры sklearn: {'poly__degree': 2}
Файл: results/lab02_results.csv
